# 02 - Feature Pipeline

Builds the model-ready feature matrix from the raw store: returns, rolling volatility, momentum, and the aligned macro block, followed by the outlier screen and the date-based train/validation/test split. Every parameter comes from `config/config.yaml` so the split dates are identical across all experiments. Outputs are written once and read by notebooks 03-05.

Two leakage guards are enforced here:

1. **Point-in-time macro data.** Most FRED series (CPIAUCSL above all) carry a publication lag. Using the same-day value would hand the model information no trader had on the bar. Macro rows are therefore shifted by a configurable release lag before joining.
2. **T+1 allocation convention.** Signals computed on day *t* are only tradeable on day *t+1*, so the LSTM targets are next-day returns; nothing on day *t*'s row looks forward within itself.

In [ ]:
import os
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd
import yaml

from src.cleaning import winsorize
from src.config import load_config, project_root

CFG = load_config()
ROOT = project_root()

random.seed(CFG["seeds"]["random"])
np.random.seed(CFG["seeds"]["numpy"])

prices = pd.read_csv(ROOT / "data/raw/prices.csv", index_col=0, parse_dates=True)
macro = pd.read_csv(ROOT / "data/raw/macro.csv", index_col=0, parse_dates=True)

# Calendar alignment across tickers (bounded ffill per the cleaning policy).
prices = prices.sort_index().ffill(limit=CFG["cleaning"]["ffill_limit"]).dropna(how="all")
print("Aligned price matrix:", prices.shape)

## 1. Core features

Daily simple returns feed the LSTM directly; annualized rolling volatility and multi-horizon momentum capture the persistence the networks are expected to exploit (Problem Statement, Section 3.1).

All features on row *t* use data up to and including the close of day *t* only. The supervised target is the **next-day** return (`ret_next`), consistent with the T+1 allocation convention.

In [ ]:
rets = prices.pct_change().dropna()
vol = rets.rolling(CFG["features"]["vol_window"]).std() * np.sqrt(252)
mom = rets.rolling(CFG["features"]["mom_window"]).sum()

# Target: next-day return, aligned so row t carries E[r_{t+1} | info_t].
ret_next = rets.shift(-1)

# Macro block: point-in-time shift for publication lag, then align to the
# trading calendar. Default lag of 14 business days covers CPIAUCSL (the
# slowest series in the set); fast series (VIXCLS, DFF) are checked again
# against their real release calendars at M3 and given per-series lags.
MACRO_LAG_BDAYS = int(CFG["features"].get("macro_lag_bdays", 14))
macro = macro.sort_index().ffill(limit=CFG["cleaning"]["ffill_limit"])
macro = macro.shift(MACRO_LAG_BDAYS).reindex(rets.index).dropna(how="all")

features = pd.concat({"ret": rets, "vol": vol, "mom": mom}, axis=1)
targets = pd.concat({"ret_next": ret_next}, axis=1)
print("Feature blocks:", features.columns.get_level_values(0).unique().tolist())
print("Macro series aligned (lagged):", macro.columns.tolist())

## 2. Outlier screen

Winsorization at the configured quantile, with the rule fixed before the test block is touched (pre-commitment against data snooping).

The clip is applied **per column**: a single cross-sectional bound would let one volatile name drag the tails of every quiet sector ETF toward its own distribution.

In [ ]:
q_lo, q_hi = CFG["cleaning"]["winsor"]
screened = winsorize(features, q_lo, q_hi)
targets = winsorize(targets, q_lo, q_hi)   # same rule on the label side
n_clipped = int((features != screened).sum().sum())
print("Values clipped:", n_clipped)

## 3. Date-based splits

Train/validation/test boundaries come from the config, not from data. The test block remains untouched until the final walk-forward run.

In [ ]:
splits = CFG["splits"]
train = screened.loc[splits["train_start"]:splits["train_end"]]
valid = screened.loc[splits["valid_start"]:splits["valid_end"]]
test = screened.loc[splits["test_start"]:]
for name, frame in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name}: {frame.index.min().date()} -> {frame.index.max().date()}  rows={len(frame)}")
assert not test.empty, "Test block empty - check split dates in config."

## 4. Persist

Parquet is used for the feature matrix to preserve dtypes. The cleaning log below is the artifact the report cites.

In [ ]:
os.makedirs(ROOT / "data/processed", exist_ok=True)
screened.to_parquet(ROOT / "data/processed/features.parquet")
targets.to_parquet(ROOT / "data/processed/targets.parquet")
macro.to_csv(ROOT / "data/processed/macro_aligned.csv")
log = pd.DataFrame({
    "step": ["ffill_limit", "winsor_low", "winsor_high", "macro_lag_bdays",
             "values_clipped", "rows", "tickers"],
    "value": [CFG["cleaning"]["ffill_limit"], q_lo, q_hi, MACRO_LAG_BDAYS,
              n_clipped, len(screened), prices.shape[1]],
})
log.to_csv(ROOT / "data/processed/cleaning_log.csv", index=False)
print(log)